# Option-Conditioned Carry Strategy

This notebook extends the baseline monthly carry strategy from `04_em_carry_extension.ipynb` using the option-surface findings from `06_options_filter_regression.ipynb`.

The notebook intentionally uses the **same primary currency panel and canonical variable definitions as Notebook 06**. Universe labels are kept simple:

- **All**
- **G10**
- **EM**

The baseline portfolio logic remains transparent: long high-carry currencies and short low-carry currencies. The research question is whether option information improves net risk-adjusted performance without relying on in-sample coefficient significance alone.

## Research hypotheses carried forward from Notebook 06

Notebook 06 motivates a deliberately small strategy ladder:

1. **Baseline carry:** reproduce the transparent long-high / short-low carry portfolio on the Notebook 06 panel.
2. **G10 ATM-conditioned carry:** test whether the carry score should be strengthened when a G10 currency has relatively high ATM implied volatility.
3. **Option-risk-scaled carry:** use ATM volatility and butterfly primarily as ex-ante risk controls, because their evidence for future return magnitude is stronger than their evidence for mean return.
4. **Hybrid conditional carry:** combine the G10 interaction hypothesis with option-based risk scaling.
5. **Expanding M5 ridge score:** use a fully out-of-sample, regularized version of the joint regression score as a diagnostic benchmark.

Depreciation skew is retained in the full M5 benchmark, but it is **not** used as a hard primary filter because its carry interaction was not robust in Notebook 06.

The notebook does not choose a strategy because it has the most stars in a regression table. It evaluates:

- net return and Sharpe;
- drawdown and downside risk;
- turnover and implementation costs;
- subperiod stability;
- baseline-relative improvement;
- sensitivity to predeclared parameters.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
except Exception as exc:
    plt = None
    PLOTTING_AVAILABLE = False
    warnings.warn(f"Matplotlib unavailable: {exc}")

try:
    import statsmodels.api as sm
    STATSMODELS_AVAILABLE = True
except Exception as exc:
    sm = None
    STATSMODELS_AVAILABLE = False
    warnings.warn(f"Statsmodels unavailable: {exc}")

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = str

MONTHS_PER_YEAR = 12
MIN_MODEL_MONTHS = 60
RIDGE_ALPHA = 10.0
TARGET_VOL_ANNUAL = 0.10
VOL_LOOKBACK_MONTHS = 36
VOL_SCALE_MIN = 0.50
VOL_SCALE_MAX = 2.00

# Primary strategy settings are predeclared rather than selected on the full sample.
PRIMARY_ATM_INTERACTION_STRENGTH = 0.50
PRIMARY_ATM_RISK_PENALTY = 0.35
PRIMARY_BUTTERFLY_RISK_PENALTY = 0.15

# Cost assumptions are editable research inputs, not empirical claims.
COST_SCENARIOS = {
    "Low cost": {
        "roll_bps": {"G10": 1.0, "EM": 5.0},
        "turnover_bps": {"G10": 0.5, "EM": 2.5},
    },
    "Base cost": {
        "roll_bps": {"G10": 2.0, "EM": 10.0},
        "turnover_bps": {"G10": 1.0, "EM": 5.0},
    },
    "High cost": {
        "roll_bps": {"G10": 5.0, "EM": 25.0},
        "turnover_bps": {"G10": 2.5, "EM": 12.5},
    },
}
PRIMARY_COST_SCENARIO = "Base cost"

CRISIS_WINDOWS = [
    {"period": "2008 global financial crisis", "start": "2008-07-31", "end": "2009-03-31"},
    {"period": "2013 taper tantrum", "start": "2013-05-31", "end": "2013-09-30"},
    {"period": "2015 EM / China stress", "start": "2015-06-30", "end": "2016-02-29"},
    {"period": "2020 Covid shock", "start": "2020-02-29", "end": "2020-04-30"},
    {"period": "2022 dollar / rates shock", "start": "2022-01-31", "end": "2022-10-31"},
]


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "theo" / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root containing data/ and theo/data/processed/."
    )


ROOT = find_project_root()
PROCESSED_DIR = ROOT / "theo" / "data" / "processed"
FIG_DIR = PROCESSED_DIR / "option_conditioned_carry_strategy_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "primary_panel": PROCESSED_DIR / "option_filter_regression_panel_primary_v9.parquet",
    "regression_summary": PROCESSED_DIR / "option_filter_regression_summary_v9.json",
}

OUTPUTS = {
    "weights": PROCESSED_DIR / "option_conditioned_carry_weights.parquet",
    "returns": PROCESSED_DIR / "option_conditioned_carry_returns.parquet",
    "performance": PROCESSED_DIR / "option_conditioned_carry_performance.parquet",
    "baseline_comparison": PROCESSED_DIR / "option_conditioned_carry_baseline_comparison.parquet",
    "parameter_sensitivity": PROCESSED_DIR / "option_conditioned_carry_parameter_sensitivity.parquet",
    "composition": PROCESSED_DIR / "option_conditioned_carry_composition.parquet",
    "crisis": PROCESSED_DIR / "option_conditioned_carry_crisis_results.parquet",
    "validation": PROCESSED_DIR / "option_conditioned_carry_validation.json",
    "summary": PROCESSED_DIR / "option_conditioned_carry_summary.json",
}

for input_path in INPUTS.values():
    if input_path.resolve() in {p.resolve() for p in OUTPUTS.values()}:
        raise AssertionError(f"Input/output path collision: {input_path}")

print(f"Project root: {ROOT}")
print(f"Primary panel: {INPUTS['primary_panel'].relative_to(ROOT)}")

In [ ]:
if not INPUTS["primary_panel"].exists():
    candidates = sorted(
        PROCESSED_DIR.glob("option_filter_regression_panel_primary_v*.parquet"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            "No Notebook 06 primary panel was found. Run the current Notebook 06 first."
        )
    INPUTS["primary_panel"] = candidates[0]
    warnings.warn(f"Using latest available Notebook 06 primary panel: {candidates[0].name}")

panel = pd.read_parquet(INPUTS["primary_panel"]).copy()
panel["month_end"] = pd.to_datetime(panel["month_end"], errors="coerce")
panel["currency"] = panel["currency"].astype(str)
panel["asset_class"] = panel["asset_class"].astype(str)
panel = panel.sort_values(["month_end", "asset_class", "currency"], kind="mergesort").reset_index(drop=True)

required_columns = [
    "month_end",
    "currency",
    "asset_class",
    "carry_signal_1m",
    "realized_return_1m",
    "atm_vol_1m",
    "depreciation_skew_25d_1m",
    "bf25_1m",
]
missing = [c for c in required_columns if c not in panel.columns]
if missing:
    raise KeyError(f"Notebook 06 primary panel is missing required columns: {missing}")

if panel.duplicated(["month_end", "currency"]).any():
    raise ValueError("Duplicate month_end × currency rows detected.")

allowed_classes = {"G10", "EM"}
unexpected_classes = set(panel["asset_class"].dropna().unique()) - allowed_classes
if unexpected_classes:
    raise ValueError(f"Unexpected asset classes: {sorted(unexpected_classes)}")

regression_summary = {}
if INPUTS["regression_summary"].exists():
    regression_summary = json.loads(INPUTS["regression_summary"].read_text())

universe_summary = (
    panel.groupby("asset_class", as_index=False)
    .agg(
        observations=("currency", "size"),
        currencies=("currency", "nunique"),
        first_month=("month_end", "min"),
        last_month=("month_end", "max"),
        valid_returns=("realized_return_1m", "count"),
    )
)
all_row = pd.DataFrame([{
    "asset_class": "All",
    "observations": len(panel),
    "currencies": panel["currency"].nunique(),
    "first_month": panel["month_end"].min(),
    "last_month": panel["month_end"].max(),
    "valid_returns": panel["realized_return_1m"].notna().sum(),
}])
universe_summary = pd.concat([all_row, universe_summary], ignore_index=True)

display(Markdown("### Strategy universe inherited from Notebook 06"))
display(universe_summary)

print(
    f"Loaded {len(panel):,} rows, "
    f"{panel['currency'].nunique()} currencies, "
    f"{panel['month_end'].nunique()} months."
)

## Canonical features and no-look-ahead design

The saved Notebook 06 panel should already contain monthly within-asset-class transforms. This notebook validates them and reconstructs them only if necessary.

For the option-conditioned score, define:

\[
C_{i,t}=z(\text{carry}_{i,t}),\qquad
V_{i,t}=z(\text{ATM vol}_{i,t}),\qquad
D_{i,t}=z(\text{depreciation skew}_{i,t}),\qquad
B_{i,t}=z(\text{butterfly}_{i,t}).
\]

The fixed G10 interaction score is:

\[
Score^{G10}_{i,t}=C_{i,t}\left(1+\lambda V_{i,t}\right),
\]

with a clipped multiplier so that an extreme observation cannot create an uncontrolled sign flip or leverage spike.

The option-risk multiplier is:

\[
RiskMultiplier_{i,t}
=
\exp\left[
-\gamma V_{i,t}
-\eta \max(B_{i,t},0)
\right],
\]

then clipped and renormalized within each long and short leg. This uses ATM volatility and butterfly as risk controls rather than assuming they are standalone alpha signals.

In [ ]:
def safe_group_zscore(series, min_n=3):
    x = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=x.index, dtype=float)
    valid = x.notna()
    if valid.sum() < min_n:
        return out
    std = x[valid].std(ddof=0)
    if not np.isfinite(std) or std <= 0:
        return out
    out.loc[valid] = (x[valid] - x[valid].mean()) / std
    return out


canonical_z_columns = {
    "z_carry_class": "carry_signal_1m",
    "z_atm_vol_class": "atm_vol_1m",
    "z_depreciation_skew_class": "depreciation_skew_25d_1m",
    "z_bf25_class": "bf25_1m",
}

for z_col, raw_col in canonical_z_columns.items():
    if z_col not in panel.columns:
        panel[z_col] = (
            panel.groupby(["month_end", "asset_class"], group_keys=False)[raw_col]
            .apply(safe_group_zscore)
        )

feature_map = {
    "carry": "z_carry_class",
    "atm_vol": "z_atm_vol_class",
    "depreciation_skew": "z_depreciation_skew_class",
    "bf25": "z_bf25_class",
}

panel["carry_x_atm_vol"] = panel["z_carry_class"] * panel["z_atm_vol_class"]
panel["carry_x_depreciation_skew"] = panel["z_carry_class"] * panel["z_depreciation_skew_class"]
panel["carry_x_bf25"] = panel["z_carry_class"] * panel["z_bf25_class"]

M5_FEATURES = [
    "z_carry_class",
    "z_atm_vol_class",
    "z_depreciation_skew_class",
    "z_bf25_class",
    "carry_x_atm_vol",
    "carry_x_depreciation_skew",
    "carry_x_bf25",
]

transform_audit = []
for asset_class, sub in panel.groupby("asset_class", sort=True):
    for col in canonical_z_columns:
        monthly_means = sub.groupby("month_end")[col].mean()
        monthly_stds = sub.groupby("month_end")[col].std(ddof=0)
        transform_audit.append({
            "asset_class": asset_class,
            "variable": col,
            "available": int(sub[col].notna().sum()),
            "mean_abs_monthly_mean": float(monthly_means.abs().mean()),
            "mean_monthly_std": float(monthly_stds.mean()),
        })

transform_audit = pd.DataFrame(transform_audit)
display(Markdown("### Cross-sectional feature audit"))
display(transform_audit.style.format({
    "mean_abs_monthly_mean": "{:.3e}",
    "mean_monthly_std": "{:.3f}",
}))

## Expanding M5 ridge benchmark

The full M5 score is included as a diagnostic, but coefficients are never estimated with future observations.

For each asset class and rebalance month:

1. use only observations whose signal month is earlier than the current month;
2. require at least 60 prior months;
3. fit a ridge-regularized linear model with an unpenalized intercept;
4. predict the current cross-section;
5. use predicted return only for ranking.

Regularization is used because the option signals and interactions are correlated, while the cross-section contains relatively few currencies.

In [ ]:
def ridge_fit(train_X, train_y, alpha):
    X = np.asarray(train_X, dtype=float)
    y = np.asarray(train_y, dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    penalty = np.eye(X_design.shape[1]) * float(alpha)
    penalty[0, 0] = 0.0
    lhs = X_design.T @ X_design + penalty
    rhs = X_design.T @ y
    beta = np.linalg.solve(lhs, rhs)
    return beta


def ridge_predict(beta, test_X):
    X = np.asarray(test_X, dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    return X_design @ beta


def expanding_m5_predictions(data, min_months=60, alpha=10.0):
    output = pd.Series(np.nan, index=data.index, dtype=float)
    diagnostics = []

    for asset_class, class_data in data.groupby("asset_class", sort=True):
        class_data = class_data.sort_values(["month_end", "currency"], kind="mergesort")
        months = list(pd.Index(class_data["month_end"].dropna().unique()).sort_values())

        for current_month in months:
            prior_months = [m for m in months if m < current_month]
            if len(prior_months) < min_months:
                continue

            train = class_data[class_data["month_end"] < current_month].dropna(
                subset=M5_FEATURES + ["realized_return_1m"]
            )
            test = class_data[class_data["month_end"] == current_month].dropna(
                subset=M5_FEATURES
            )
            if test.empty or len(train) <= len(M5_FEATURES) + 10:
                continue

            X_train = train[M5_FEATURES].to_numpy(dtype=float)
            y_train = train["realized_return_1m"].to_numpy(dtype=float)
            X_test = test[M5_FEATURES].to_numpy(dtype=float)

            if not (
                np.isfinite(X_train).all()
                and np.isfinite(y_train).all()
                and np.isfinite(X_test).all()
            ):
                raise ValueError(
                    f"Non-finite M5 design values for {asset_class} at {current_month}."
                )

            beta = ridge_fit(X_train, y_train, alpha=alpha)
            output.loc[test.index] = ridge_predict(beta, X_test)

            diagnostics.append({
                "asset_class": asset_class,
                "month_end": current_month,
                "training_months": len(pd.Index(train["month_end"].unique())),
                "training_rows": len(train),
                "test_rows": len(test),
                "ridge_alpha": alpha,
                "condition_number": float(np.linalg.cond(
                    np.column_stack([np.ones(len(X_train)), X_train])
                )),
            })

    return output, pd.DataFrame(diagnostics)


panel["m5_ridge_predicted_return"] , ridge_diagnostics = expanding_m5_predictions(
    panel,
    min_months=MIN_MODEL_MONTHS,
    alpha=RIDGE_ALPHA,
)

display(Markdown("### Expanding M5 ridge diagnostics"))
display(
    ridge_diagnostics.groupby("asset_class", as_index=False).agg(
        prediction_months=("month_end", "nunique"),
        first_prediction=("month_end", "min"),
        last_prediction=("month_end", "max"),
        average_training_rows=("training_rows", "mean"),
        max_condition_number=("condition_number", "max"),
    ) if not ridge_diagnostics.empty else ridge_diagnostics
)

## Strategy specifications

The main comparison is intentionally compact.

| Strategy | Ranking signal | Position weighting |
|---|---|---|
| Baseline carry | Carry | Equal weight within long/short legs |
| G10 ATM-conditioned carry | \(C(1+\lambda V)\) in G10; carry in EM | Equal weight |
| Option-risk-scaled carry | Carry | Penalize high ATM vol and positive butterfly |
| Hybrid conditional carry | G10 ATM-conditioned score; carry in EM | Option-risk-scaled weights |
| Expanding M5 ridge score | Expanding predicted return | Option-risk-scaled weights |

For **All**, G10 and EM are constructed as separate sleeves and assigned equal capital. This matches the within-asset-class research design in Notebook 06 and prevents the combined portfolio from becoming a disguised ranking of one asset class against the other.

In [ ]:
def choose_leg_size(n):
    """Reproduce the transparent adaptive leg-size rule from Notebook 04."""
    if n < 2:
        return 0
    if n >= 6:
        return min(max(math.ceil(0.20 * n), 3), n // 2)
    return max(1, n // 3)


STRATEGY_SPECS = {
    "Baseline carry": {
        "score_type": "carry",
        "atm_interaction_strength": 0.0,
        "atm_risk_penalty": 0.0,
        "butterfly_risk_penalty": 0.0,
    },
    "G10 ATM-conditioned carry": {
        "score_type": "carry",
        "atm_interaction_strength": PRIMARY_ATM_INTERACTION_STRENGTH,
        "atm_risk_penalty": 0.0,
        "butterfly_risk_penalty": 0.0,
    },
    "Option-risk-scaled carry": {
        "score_type": "carry",
        "atm_interaction_strength": 0.0,
        "atm_risk_penalty": PRIMARY_ATM_RISK_PENALTY,
        "butterfly_risk_penalty": PRIMARY_BUTTERFLY_RISK_PENALTY,
    },
    "Hybrid conditional carry": {
        "score_type": "carry",
        "atm_interaction_strength": PRIMARY_ATM_INTERACTION_STRENGTH,
        "atm_risk_penalty": PRIMARY_ATM_RISK_PENALTY,
        "butterfly_risk_penalty": PRIMARY_BUTTERFLY_RISK_PENALTY,
    },
    "Expanding M5 ridge score": {
        "score_type": "m5_ridge",
        "atm_interaction_strength": 0.0,
        "atm_risk_penalty": PRIMARY_ATM_RISK_PENALTY,
        "butterfly_risk_penalty": PRIMARY_BUTTERFLY_RISK_PENALTY,
    },
}


def prepare_strategy_signals(data, spec):
    out = data.copy()
    carry = out["z_carry_class"]
    atm = out["z_atm_vol_class"].clip(-3.0, 3.0)
    butterfly = out["z_bf25_class"].clip(-3.0, 3.0)

    if spec["score_type"] == "m5_ridge":
        score = out["m5_ridge_predicted_return"].copy()
    else:
        score = carry.copy()
        strength = float(spec["atm_interaction_strength"])
        if strength != 0:
            # Apply the interaction hypothesis only to G10.
            condition_multiplier = (1.0 + strength * atm).clip(0.25, 2.00)
            score = np.where(
                out["asset_class"].eq("G10"),
                carry * condition_multiplier,
                carry,
            )
            score = pd.Series(score, index=out.index, dtype=float)

    gamma = float(spec["atm_risk_penalty"])
    eta = float(spec["butterfly_risk_penalty"])
    risk_multiplier = np.exp(
        -gamma * atm
        -eta * butterfly.clip(lower=0.0)
    )
    risk_multiplier = pd.Series(
        np.clip(risk_multiplier, 0.35, 2.00),
        index=out.index,
        dtype=float,
    )

    out["strategy_score"] = score
    out["risk_multiplier"] = risk_multiplier
    return out


def build_sleeve_positions(month_data, allocation):
    # Portfolio membership is determined only from information available at the
    # signal date. Future realized returns must never determine eligibility.
    sub = month_data.dropna(
        subset=["strategy_score", "risk_multiplier"]
    ).copy()
    sub = sub.sort_values(["strategy_score", "currency"], kind="mergesort")
    n = len(sub)
    k = choose_leg_size(n)
    if k == 0 or n < 2 * k:
        return pd.DataFrame()

    low = sub.head(k).copy()
    high = sub.tail(k).copy()

    high_risk = high["risk_multiplier"].clip(lower=1e-8)
    low_risk = low["risk_multiplier"].clip(lower=1e-8)

    high["weight"] = allocation * high_risk / high_risk.sum()
    low["weight"] = -allocation * low_risk / low_risk.sum()

    high["basket"] = "long_high_score"
    low["basket"] = "short_low_score"

    positions = pd.concat([low, high], ignore_index=False)
    positions["n_universe"] = n
    positions["leg_size"] = k
    return positions


def build_strategy_weights(data, strategy_name, spec, sample):
    scored = prepare_strategy_signals(data, spec)

    if sample == "G10":
        sample_data = scored[scored["asset_class"].eq("G10")].copy()
        class_allocations = {"G10": 1.0}
    elif sample == "EM":
        sample_data = scored[scored["asset_class"].eq("EM")].copy()
        class_allocations = {"EM": 1.0}
    elif sample == "All":
        sample_data = scored.copy()
        class_allocations = {"G10": 0.5, "EM": 0.5}
    else:
        raise ValueError(f"Unknown sample: {sample}")

    rows = []
    for month_end, month_data in sample_data.groupby("month_end", sort=True):
        for asset_class, allocation in class_allocations.items():
            sleeve = month_data[month_data["asset_class"].eq(asset_class)]
            positions = build_sleeve_positions(sleeve, allocation=allocation)
            if positions.empty:
                continue
            positions["month_end"] = month_end
            positions["strategy"] = strategy_name
            positions["sample"] = sample
            positions["class_allocation"] = allocation
            rows.append(positions)

    if not rows:
        return pd.DataFrame()

    weights = pd.concat(rows, ignore_index=True)
    weights = weights.sort_values(
        ["sample", "strategy", "month_end", "asset_class", "currency"],
        kind="mergesort",
    )
    return weights


weight_parts = []
for sample in ["All", "G10", "EM"]:
    for strategy_name, spec in STRATEGY_SPECS.items():
        result = build_strategy_weights(panel, strategy_name, spec, sample)
        if not result.empty:
            weight_parts.append(result)

strategy_weights = (
    pd.concat(weight_parts, ignore_index=True)
    if weight_parts else pd.DataFrame()
)

display(Markdown("### Strategy availability"))
display(
    strategy_weights.groupby(["sample", "strategy"], as_index=False).agg(
        months=("month_end", "nunique"),
        first_month=("month_end", "min"),
        last_month=("month_end", "max"),
        selected_currency_rows=("currency", "count"),
    ) if not strategy_weights.empty else strategy_weights
)

## Returns, turnover, and implementation costs

Each single-asset-class portfolio has:

- long-leg notional \(+1\);
- short-leg notional \(-1\);
- gross notional \(2\);
- net notional \(0\).

The **All** portfolio allocates half of capital to each asset-class sleeve, leaving the same total gross notional.

Net return is calculated as:

\[
r^{net}_{t+1}
=
r^{gross}_{t+1}
-
\sum_i |w_{i,t}|\cdot RollCost_i
-
\sum_i |w_{i,t}-w_{i,t-1}|\cdot TradingCost_i.
\]

Costs are reported under low, base, and high scenarios. The assumptions are centralized in the configuration cell so they can be changed without changing strategy logic.

In [ ]:
def add_turnover(weights):
    if weights.empty:
        return weights.copy()

    out = weights.copy()
    keys = ["sample", "strategy", "asset_class", "currency"]
    out = out.sort_values(keys + ["month_end"], kind="mergesort")
    out["previous_weight"] = (
        out.groupby(keys, sort=False)["weight"].shift(1).fillna(0.0)
    )

    # Include currencies that leave the portfolio by using a full monthly weight grid.
    rebuilt = []
    for (sample, strategy), sub in out.groupby(["sample", "strategy"], sort=True):
        months = pd.Index(sub["month_end"].unique()).sort_values()
        currencies = (
            sub[["currency", "asset_class"]]
            .drop_duplicates()
            .sort_values(["asset_class", "currency"])
        )
        grid = (
            pd.MultiIndex.from_product(
                [months, currencies["currency"]],
                names=["month_end", "currency"],
            )
            .to_frame(index=False)
            .merge(currencies, on="currency", how="left", validate="many_to_one")
        )
        existing = sub.drop(columns=["previous_weight"], errors="ignore")
        grid = grid.merge(
            existing,
            on=["month_end", "currency", "asset_class"],
            how="left",
            validate="one_to_one",
        )
        grid["sample"] = sample
        grid["strategy"] = strategy
        grid["weight"] = grid["weight"].fillna(0.0)
        grid = grid.sort_values(["currency", "month_end"], kind="mergesort")
        grid["previous_weight"] = (
            grid.groupby("currency", sort=False)["weight"].shift(1).fillna(0.0)
        )
        grid["absolute_turnover"] = (grid["weight"] - grid["previous_weight"]).abs()
        rebuilt.append(grid)

    return pd.concat(rebuilt, ignore_index=True)


weights_with_turnover = add_turnover(strategy_weights)


def calculate_strategy_returns(weights):
    rows = []
    for (sample, strategy, month_end), sub in weights.groupby(
        ["sample", "strategy", "month_end"], sort=True
    ):
        active = sub[sub["weight"].ne(0)].copy()
        missing_selected_returns = int(active["realized_return_1m"].isna().sum())
        gross_return = (
            np.nan
            if missing_selected_returns > 0
            else float((active["weight"] * active["realized_return_1m"]).sum())
        )

        row = {
            "sample": sample,
            "strategy": strategy,
            "month_end": month_end,
            "gross_return": gross_return,
            "gross_notional": active["weight"].abs().sum(),
            "net_notional": active["weight"].sum(),
            "turnover": sub["absolute_turnover"].sum(),
            "n_active": active["currency"].nunique(),
            "missing_selected_returns": missing_selected_returns,
            "long_basket": ", ".join(sorted(
                active.loc[active["weight"] > 0, "currency"].astype(str)
            )),
            "short_basket": ", ".join(sorted(
                active.loc[active["weight"] < 0, "currency"].astype(str)
            )),
        }

        for scenario, assumptions in COST_SCENARIOS.items():
            roll_cost = 0.0
            turnover_cost = 0.0
            for asset_class, class_sub in sub.groupby("asset_class", sort=False):
                roll_bps = assumptions["roll_bps"][asset_class]
                turnover_bps = assumptions["turnover_bps"][asset_class]
                roll_cost += (
                    class_sub["weight"].abs().sum() * roll_bps / 10000.0
                )
                turnover_cost += (
                    class_sub["absolute_turnover"].sum()
                    * turnover_bps / 10000.0
                )

            safe_name = scenario.lower().replace(" ", "_")
            row[f"roll_cost_{safe_name}"] = roll_cost
            row[f"turnover_cost_{safe_name}"] = turnover_cost
            row[f"net_return_{safe_name}"] = (
                gross_return - roll_cost - turnover_cost
            )

        rows.append(row)

    returns = pd.DataFrame(rows)
    return returns.sort_values(
        ["sample", "strategy", "month_end"], kind="mergesort"
    )


strategy_returns = calculate_strategy_returns(weights_with_turnover)
PRIMARY_NET_COL = f"net_return_{PRIMARY_COST_SCENARIO.lower().replace(' ', '_')}"

# Optional portfolio-level volatility targeting using only prior realized returns.
strategy_returns["ex_ante_vol"] = np.nan
strategy_returns["vol_scale"] = np.nan
strategy_returns["vol_targeted_net_return"] = np.nan

for (sample, strategy), idx in strategy_returns.groupby(
    ["sample", "strategy"], sort=False
).groups.items():
    sub = strategy_returns.loc[idx].sort_values("month_end")
    rolling_vol = (
        sub[PRIMARY_NET_COL]
        .rolling(VOL_LOOKBACK_MONTHS, min_periods=12)
        .std(ddof=1)
        .shift(1)
        * np.sqrt(MONTHS_PER_YEAR)
    )
    scale = (TARGET_VOL_ANNUAL / rolling_vol).clip(
        VOL_SCALE_MIN, VOL_SCALE_MAX
    )
    strategy_returns.loc[sub.index, "ex_ante_vol"] = rolling_vol
    strategy_returns.loc[sub.index, "vol_scale"] = scale
    strategy_returns.loc[sub.index, "vol_targeted_net_return"] = (
        scale * sub[PRIMARY_NET_COL]
    )

display(Markdown("### Return and cost audit"))
display(
    strategy_returns.groupby(["sample", "strategy"], as_index=False).agg(
        months=("month_end", "count"),
        average_gross_notional=("gross_notional", "mean"),
        average_net_notional=("net_notional", "mean"),
        average_turnover=("turnover", "mean"),
        average_base_net_return=(PRIMARY_NET_COL, "mean"),
    )
)

In [ ]:
def max_drawdown(returns):
    r = pd.Series(returns).dropna().astype(float)
    if r.empty:
        return np.nan
    wealth = (1.0 + r).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())


def expected_shortfall(returns, q=0.05):
    r = pd.Series(returns).dropna().astype(float)
    if r.empty:
        return np.nan
    threshold = r.quantile(q)
    tail = r[r <= threshold]
    return float(tail.mean()) if not tail.empty else np.nan


def performance_stats(sample, strategy, return_type, returns):
    r = pd.Series(returns).dropna().astype(float)
    if r.empty:
        return {
            "sample": sample,
            "strategy": strategy,
            "return_type": return_type,
            "months": 0,
        }

    ann_mean = r.mean() * MONTHS_PER_YEAR
    ann_vol = r.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)
    downside = r[r < 0].std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)
    annualized_geometric = (1.0 + r).prod() ** (
        MONTHS_PER_YEAR / len(r)
    ) - 1.0

    return {
        "sample": sample,
        "strategy": strategy,
        "return_type": return_type,
        "months": len(r),
        "annualized_arithmetic_return": ann_mean,
        "annualized_geometric_return": annualized_geometric,
        "annualized_volatility": ann_vol,
        "sharpe_ratio": ann_mean / ann_vol if ann_vol > 0 else np.nan,
        "sortino_ratio": ann_mean / downside if downside > 0 else np.nan,
        "max_drawdown": max_drawdown(r),
        "calmar_ratio": (
            annualized_geometric / abs(max_drawdown(r))
            if np.isfinite(max_drawdown(r)) and max_drawdown(r) < 0
            else np.nan
        ),
        "skewness": r.skew(),
        "hit_rate": (r > 0).mean(),
        "worst_month": r.min(),
        "expected_shortfall_5pct": expected_shortfall(r, 0.05),
        "final_cumulative_return": (1.0 + r).prod() - 1.0,
    }


performance_rows = []
return_columns = {
    "Gross": "gross_return",
    "Low-cost net": "net_return_low_cost",
    "Base-cost net": "net_return_base_cost",
    "High-cost net": "net_return_high_cost",
    "Base-cost net, 10% vol target": "vol_targeted_net_return",
}

for (sample, strategy), sub in strategy_returns.groupby(
    ["sample", "strategy"], sort=True
):
    for return_type, column in return_columns.items():
        performance_rows.append(
            performance_stats(sample, strategy, return_type, sub[column])
        )

performance = pd.DataFrame(performance_rows)

primary_performance = performance[
    performance["return_type"].eq("Base-cost net")
].copy()

display(Markdown("### Base-cost net performance"))
display(
    primary_performance.sort_values(
        ["sample", "sharpe_ratio"], ascending=[True, False]
    ).style.format({
        "annualized_arithmetic_return": "{:.2%}",
        "annualized_geometric_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "sharpe_ratio": "{:.2f}",
        "sortino_ratio": "{:.2f}",
        "max_drawdown": "{:.2%}",
        "calmar_ratio": "{:.2f}",
        "skewness": "{:.2f}",
        "hit_rate": "{:.2%}",
        "worst_month": "{:.2%}",
        "expected_shortfall_5pct": "{:.2%}",
        "final_cumulative_return": "{:.2%}",
    })
)

## Baseline-relative evaluation

A strategy is useful only if it improves the baseline on economically relevant dimensions.

For each sample, the notebook reports:

- change in annualized return;
- change in Sharpe;
- change in max drawdown;
- change in expected shortfall;
- change in turnover;
- a paired HAC test of the monthly net-return difference.

The HAC test is a diagnostic, not a strategy-selection rule.

In [ ]:
def hac_mean_test(series, maxlags=3):
    x = pd.Series(series).dropna().astype(float)
    if len(x) < 24 or not STATSMODELS_AVAILABLE:
        return {"mean_difference": x.mean() if len(x) else np.nan,
                "hac_t": np.nan, "hac_p": np.nan, "observations": len(x)}
    X = np.ones((len(x), 1))
    fit = sm.OLS(x.to_numpy(), X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags},
    )
    return {
        "mean_difference": float(fit.params[0]),
        "hac_t": float(fit.tvalues[0]),
        "hac_p": float(fit.pvalues[0]),
        "observations": len(x),
    }


comparison_rows = []
for sample in ["All", "G10", "EM"]:
    sample_returns = strategy_returns[
        strategy_returns["sample"].eq(sample)
    ].copy()
    baseline = sample_returns[
        sample_returns["strategy"].eq("Baseline carry")
    ][["month_end", PRIMARY_NET_COL, "turnover"]].rename(
        columns={
            PRIMARY_NET_COL: "baseline_return",
            "turnover": "baseline_turnover",
        }
    )

    baseline_perf = primary_performance[
        (primary_performance["sample"].eq(sample))
        & (primary_performance["strategy"].eq("Baseline carry"))
    ]
    if baseline_perf.empty:
        continue
    baseline_perf = baseline_perf.iloc[0]

    for strategy, sub in sample_returns.groupby("strategy", sort=True):
        if strategy == "Baseline carry":
            continue

        merged = sub[["month_end", PRIMARY_NET_COL, "turnover"]].merge(
            baseline, on="month_end", how="inner", validate="one_to_one"
        )
        merged["return_difference"] = (
            merged[PRIMARY_NET_COL] - merged["baseline_return"]
        )
        hac = hac_mean_test(merged["return_difference"])

        strategy_perf = primary_performance[
            (primary_performance["sample"].eq(sample))
            & (primary_performance["strategy"].eq(strategy))
        ]
        if strategy_perf.empty:
            continue
        strategy_perf = strategy_perf.iloc[0]

        comparison_rows.append({
            "sample": sample,
            "strategy": strategy,
            "common_months": len(merged),
            "annualized_return_difference": (
                strategy_perf["annualized_arithmetic_return"]
                - baseline_perf["annualized_arithmetic_return"]
            ),
            "sharpe_difference": (
                strategy_perf["sharpe_ratio"]
                - baseline_perf["sharpe_ratio"]
            ),
            "max_drawdown_difference": (
                strategy_perf["max_drawdown"]
                - baseline_perf["max_drawdown"]
            ),
            "expected_shortfall_difference": (
                strategy_perf["expected_shortfall_5pct"]
                - baseline_perf["expected_shortfall_5pct"]
            ),
            "average_turnover_difference": (
                merged["turnover"] - merged["baseline_turnover"]
            ).mean(),
            "monthly_mean_return_difference": hac["mean_difference"],
            "hac_t": hac["hac_t"],
            "hac_p": hac["hac_p"],
        })

baseline_comparison = pd.DataFrame(comparison_rows)

display(Markdown("### Improvement relative to baseline carry"))
display(
    baseline_comparison.sort_values(
        ["sample", "sharpe_difference"], ascending=[True, False]
    ).style.format({
        "annualized_return_difference": "{:+.2%}",
        "sharpe_difference": "{:+.2f}",
        "max_drawdown_difference": "{:+.2%}",
        "expected_shortfall_difference": "{:+.2%}",
        "average_turnover_difference": "{:+.2f}",
        "monthly_mean_return_difference": "{:+.3%}",
        "hac_t": "{:+.2f}",
        "hac_p": "{:.3f}",
    }) if not baseline_comparison.empty else baseline_comparison
)

In [ ]:
if not PLOTTING_AVAILABLE or strategy_returns.empty:
    display(Markdown("Plots skipped because plotting is unavailable."))
else:
    for sample in ["All", "G10", "EM"]:
        sub = strategy_returns[strategy_returns["sample"].eq(sample)].copy()
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(12, 6))
        for strategy, strategy_sub in sub.groupby("strategy", sort=True):
            strategy_sub = strategy_sub.sort_values("month_end")
            wealth = (1.0 + strategy_sub[PRIMARY_NET_COL].fillna(0.0)).cumprod()
            ax.plot(strategy_sub["month_end"], wealth, label=strategy)
        ax.set_title(f"{sample}: cumulative wealth under base-cost assumptions")
        ax.set_xlabel("Signal month")
        ax.set_ylabel("Cumulative wealth from 1.0")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(
            FIG_DIR / f"{sample.lower()}_base_cost_cumulative_wealth.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.show()

        fig, ax = plt.subplots(figsize=(12, 5))
        for strategy, strategy_sub in sub.groupby("strategy", sort=True):
            strategy_sub = strategy_sub.sort_values("month_end")
            wealth = (1.0 + strategy_sub[PRIMARY_NET_COL].fillna(0.0)).cumprod()
            drawdown = wealth / wealth.cummax() - 1.0
            ax.plot(strategy_sub["month_end"], drawdown, label=strategy)
        ax.set_title(f"{sample}: drawdown under base-cost assumptions")
        ax.set_xlabel("Signal month")
        ax.set_ylabel("Drawdown")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(
            FIG_DIR / f"{sample.lower()}_base_cost_drawdown.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.show()

## Parameter sensitivity

The primary parameters were fixed before examining this strategy notebook. Sensitivity analysis checks whether the conclusion depends on one arbitrary value.

The notebook varies:

- G10 ATM interaction strength \(\lambda\);
- ATM risk penalty \(\gamma\);
- butterfly risk penalty \(\eta\).

The sensitivity table is descriptive. It should not be used to select the best full-sample combination and then report that result as out of sample.

In [ ]:
SENSITIVITY_SPECS = []

for lam in [0.00, 0.25, 0.50, 0.75, 1.00]:
    SENSITIVITY_SPECS.append({
        "sensitivity_family": "G10 ATM interaction",
        "label": f"lambda={lam:.2f}",
        "score_type": "carry",
        "atm_interaction_strength": lam,
        "atm_risk_penalty": 0.0,
        "butterfly_risk_penalty": 0.0,
        "lambda": lam,
        "gamma": 0.0,
        "eta": 0.0,
    })

for gamma in [0.00, 0.25, 0.50, 0.75]:
    for eta in [0.00, 0.15, 0.30]:
        SENSITIVITY_SPECS.append({
            "sensitivity_family": "Option-risk scaling",
            "label": f"gamma={gamma:.2f}, eta={eta:.2f}",
            "score_type": "carry",
            "atm_interaction_strength": 0.0,
            "atm_risk_penalty": gamma,
            "butterfly_risk_penalty": eta,
            "lambda": 0.0,
            "gamma": gamma,
            "eta": eta,
        })


sensitivity_rows = []
for sample in ["G10", "EM"]:
    for spec in SENSITIVITY_SPECS:
        strategy_name = f"Sensitivity: {spec['sensitivity_family']} | {spec['label']}"
        weights = build_strategy_weights(panel, strategy_name, spec, sample)
        if weights.empty:
            continue
        returns = calculate_strategy_returns(add_turnover(weights))
        perf = performance_stats(
            sample,
            strategy_name,
            "Base-cost net",
            returns[PRIMARY_NET_COL],
        )
        sensitivity_rows.append({
            **perf,
            "sensitivity_family": spec["sensitivity_family"],
            "label": spec["label"],
            "lambda": spec["lambda"],
            "gamma": spec["gamma"],
            "eta": spec["eta"],
        })

parameter_sensitivity = pd.DataFrame(sensitivity_rows)

display(Markdown("### Parameter sensitivity: base-cost net performance"))
display(
    parameter_sensitivity.sort_values(
        ["sample", "sensitivity_family", "sharpe_ratio"],
        ascending=[True, True, False],
    ).style.format({
        "annualized_arithmetic_return": "{:.2%}",
        "annualized_geometric_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "sharpe_ratio": "{:.2f}",
        "sortino_ratio": "{:.2f}",
        "max_drawdown": "{:.2%}",
        "expected_shortfall_5pct": "{:.2%}",
    }) if not parameter_sensitivity.empty else parameter_sensitivity
)

## Subperiod, crisis, and composition diagnostics

A strategy improvement is not credible if it is driven by one currency, one crisis, or one subperiod.

The following sections report:

- annual and half-sample performance;
- crisis-window returns and drawdowns;
- long/short selection rates;
- average weights and turnover by currency;
- concentration warnings.

In [ ]:
def period_stats(data, start=None, end=None):
    sub = data.copy()
    if start is not None:
        sub = sub[sub["month_end"] >= pd.Timestamp(start)]
    if end is not None:
        sub = sub[sub["month_end"] <= pd.Timestamp(end)]
    return sub


subperiod_rows = []
unique_months = pd.Index(panel["month_end"].dropna().unique()).sort_values()
cutoff = unique_months[len(unique_months) // 2]

for (sample, strategy), sub in strategy_returns.groupby(
    ["sample", "strategy"], sort=True
):
    periods = {
        "Full sample": sub,
        "First half": sub[sub["month_end"] <= cutoff],
        "Second half": sub[sub["month_end"] > cutoff],
    }
    for period, period_data in periods.items():
        stats = performance_stats(
            sample, strategy, "Base-cost net", period_data[PRIMARY_NET_COL]
        )
        subperiod_rows.append({**stats, "period": period})

subperiod_performance = pd.DataFrame(subperiod_rows)

crisis_rows = []
for (sample, strategy), sub in strategy_returns.groupby(
    ["sample", "strategy"], sort=True
):
    for window in CRISIS_WINDOWS:
        crisis_sub = sub[
            (sub["month_end"] >= pd.Timestamp(window["start"]))
            & (sub["month_end"] <= pd.Timestamp(window["end"]))
        ].copy()
        r = crisis_sub[PRIMARY_NET_COL].dropna()
        crisis_rows.append({
            "sample": sample,
            "strategy": strategy,
            "period": window["period"],
            "months": len(r),
            "period_return": (1.0 + r).prod() - 1.0 if len(r) else np.nan,
            "annualized_volatility": (
                r.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)
                if len(r) > 1 else np.nan
            ),
            "max_drawdown": max_drawdown(r),
            "worst_month": r.min() if len(r) else np.nan,
        })
crisis_results = pd.DataFrame(crisis_rows)

active_weights = weights_with_turnover[
    weights_with_turnover["weight"].ne(0)
].copy()
active_weights["is_long"] = active_weights["weight"].gt(0).astype(float)
active_weights["is_short"] = active_weights["weight"].lt(0).astype(float)

composition = (
    active_weights.groupby(
        ["sample", "strategy", "asset_class", "currency"], as_index=False
    )
    .agg(
        selected_months=("month_end", "nunique"),
        long_selection_rate=("is_long", "mean"),
        short_selection_rate=("is_short", "mean"),
        average_weight=("weight", "mean"),
        average_absolute_weight=("weight", lambda x: x.abs().mean()),
        total_turnover=("absolute_turnover", "sum"),
    )
)

display(Markdown("### Half-sample performance"))
display(
    subperiod_performance[
        subperiod_performance["period"].isin(["First half", "Second half"])
    ].style.format({
        "annualized_arithmetic_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "sharpe_ratio": "{:.2f}",
        "max_drawdown": "{:.2%}",
        "expected_shortfall_5pct": "{:.2%}",
    })
)

display(Markdown("### Crisis performance"))
display(
    crisis_results.style.format({
        "period_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "max_drawdown": "{:.2%}",
        "worst_month": "{:.2%}",
    })
)

display(Markdown("### Currency composition"))
display(
    composition.sort_values(
        ["sample", "strategy", "long_selection_rate"],
        ascending=[True, True, False],
    ).style.format({
        "long_selection_rate": "{:.1%}",
        "short_selection_rate": "{:.1%}",
        "average_weight": "{:+.3f}",
        "average_absolute_weight": "{:.3f}",
        "total_turnover": "{:.2f}",
    })
)

In [ ]:
validation_rows = []

def record_check(name, passed, detail=""):
    validation_rows.append({
        "check": name,
        "status": "pass" if bool(passed) else "fail",
        "detail": detail,
    })
    if not passed:
        raise AssertionError(f"{name}: {detail}")


record_check(
    "unique month-currency panel",
    not panel.duplicated(["month_end", "currency"]).any(),
)
record_check(
    "allowed sample labels",
    set(strategy_weights["sample"].dropna().unique()).issubset({"All", "G10", "EM"}),
)
record_check(
    "finite active weights",
    np.isfinite(strategy_weights["weight"]).all(),
)
record_check(
    "finite active scores",
    np.isfinite(strategy_weights["strategy_score"]).all(),
)
record_check(
    "portfolio selection does not require future realized returns",
    True,
    "build_sleeve_positions filters only on strategy_score and risk_multiplier",
)
missing_position_months = (
    strategy_returns.loc[
        strategy_returns["missing_selected_returns"].gt(0),
        ["sample", "strategy", "month_end", "gross_return", PRIMARY_NET_COL],
    ]
)
record_check(
    "missing selected returns are never treated as zero",
    missing_position_months.empty
    or bool(
        missing_position_months["gross_return"].isna().all()
        and missing_position_months[PRIMARY_NET_COL].isna().all()
    ),
    f"strategy-months with missing selected returns={len(missing_position_months)}",
)

weight_sums = (
    strategy_weights.groupby(["sample", "strategy", "month_end"], as_index=False)
    .agg(
        net_weight=("weight", "sum"),
        gross_weight=("weight", lambda x: x.abs().sum()),
    )
)
record_check(
    "monthly net weights approximately zero",
    np.allclose(weight_sums["net_weight"], 0.0, atol=1e-10),
    f"max absolute net weight={weight_sums['net_weight'].abs().max():.3e}",
)
record_check(
    "monthly gross weights approximately two",
    np.allclose(weight_sums["gross_weight"], 2.0, atol=1e-10),
    f"range={weight_sums['gross_weight'].min():.6f} to {weight_sums['gross_weight'].max():.6f}",
)
record_check(
    "nonnegative implementation costs",
    bool(
        (
            strategy_returns.filter(regex=r"^(roll_cost|turnover_cost)_")
            .fillna(0.0)
            >= 0
        ).all().all()
    ),
)
net_cost_difference = (
    strategy_returns["gross_return"]
    - strategy_returns[PRIMARY_NET_COL]
).dropna()
record_check(
    "net return does not exceed gross by construction",
    bool((net_cost_difference >= -1e-12).all()),
)
record_check(
    "M5 predictions use delayed expanding estimation",
    ridge_diagnostics.empty
    or bool((ridge_diagnostics["training_months"] >= MIN_MODEL_MONTHS).all()),
)
record_check(
    "output paths do not overwrite Notebook 06 input",
    INPUTS["primary_panel"].resolve()
    not in {path.resolve() for path in OUTPUTS.values()},
)

validation = pd.DataFrame(validation_rows)
display(Markdown("### Strategy integrity checks"))
display(validation)

In [ ]:
strategy_weights.to_parquet(OUTPUTS["weights"], index=False)
strategy_returns.to_parquet(OUTPUTS["returns"], index=False)
performance.to_parquet(OUTPUTS["performance"], index=False)
baseline_comparison.to_parquet(OUTPUTS["baseline_comparison"], index=False)
parameter_sensitivity.to_parquet(OUTPUTS["parameter_sensitivity"], index=False)
composition.to_parquet(OUTPUTS["composition"], index=False)
crisis_results.to_parquet(OUTPUTS["crisis"], index=False)

validation_payload = {
    "source_panel": str(INPUTS["primary_panel"]),
    "checks": validation.to_dict("records"),
    "all_checks_passed": bool(validation["status"].eq("pass").all()),
}
OUTPUTS["validation"].write_text(
    json.dumps(validation_payload, indent=2, default=str)
)

summary_payload = {
    "notebook": "07_option_conditioned_carry_strategy.ipynb",
    "source_panel": str(INPUTS["primary_panel"]),
    "samples": ["All", "G10", "EM"],
    "strategies": list(STRATEGY_SPECS),
    "primary_cost_scenario": PRIMARY_COST_SCENARIO,
    "strategy_parameters": {
        "atm_interaction_strength": PRIMARY_ATM_INTERACTION_STRENGTH,
        "atm_risk_penalty": PRIMARY_ATM_RISK_PENALTY,
        "butterfly_risk_penalty": PRIMARY_BUTTERFLY_RISK_PENALTY,
        "ridge_alpha": RIDGE_ALPHA,
        "minimum_model_months": MIN_MODEL_MONTHS,
    },
    "primary_performance": primary_performance.to_dict("records"),
    "baseline_comparison": baseline_comparison.to_dict("records"),
    "validation": validation_payload,
    "output_paths": {k: str(v) for k, v in OUTPUTS.items()},
}
OUTPUTS["summary"].write_text(
    json.dumps(summary_payload, indent=2, default=str)
)

output_table = pd.DataFrame([
    {
        "output": name,
        "path": str(path.relative_to(ROOT)),
        "exists": path.exists(),
    }
    for name, path in OUTPUTS.items()
])
display(Markdown("### Saved outputs"))
display(output_table)

In [ ]:
def fmt_pct(x):
    return "n/a" if not np.isfinite(x) else f"{x:.2%}"


def fmt_num(x):
    return "n/a" if not np.isfinite(x) else f"{x:.2f}"


interpretation = [
    "## Final interpretation",
    "",
    "The strategy conclusions below are generated from base-cost net results.",
    "The fixed option rules were motivated by Notebook 06, so this is not a fully independent holdout experiment.",
]

for sample in ["All", "G10", "EM"]:
    sub = primary_performance[primary_performance["sample"].eq(sample)].copy()
    if sub.empty:
        continue
    sub = sub.sort_values("sharpe_ratio", ascending=False)
    best = sub.iloc[0]
    baseline = sub[sub["strategy"].eq("Baseline carry")]
    baseline = baseline.iloc[0] if not baseline.empty else None

    interpretation.extend([
        "",
        f"### {sample}",
        f"- Highest base-cost net Sharpe in the predefined strategy set: **{best['strategy']}** "
        f"with Sharpe {fmt_num(best['sharpe_ratio'])}, annualized arithmetic return "
        f"{fmt_pct(best['annualized_arithmetic_return'])}, and max drawdown "
        f"{fmt_pct(best['max_drawdown'])}.",
    ])

    if baseline is not None:
        interpretation.append(
            f"- Baseline carry: Sharpe {fmt_num(baseline['sharpe_ratio'])}, "
            f"annualized arithmetic return {fmt_pct(baseline['annualized_arithmetic_return'])}, "
            f"and max drawdown {fmt_pct(baseline['max_drawdown'])}."
        )

    comparison = baseline_comparison[
        (baseline_comparison["sample"].eq(sample))
        & (baseline_comparison["strategy"].eq(best["strategy"]))
    ]
    if not comparison.empty:
        row = comparison.iloc[0]
        interpretation.append(
            f"- Relative to baseline, the top predefined strategy changes Sharpe by "
            f"{row['sharpe_difference']:+.2f}, annualized return by "
            f"{row['annualized_return_difference']:+.2%}, and max drawdown by "
            f"{row['max_drawdown_difference']:+.2%}. The paired monthly HAC p-value "
            f"for the mean return difference is "
            f"{row['hac_p']:.3f}" if np.isfinite(row["hac_p"]) else
            "- Paired HAC inference is unavailable."
        )

interpretation.extend([
    "",
    "### Decision rules",
    "- Prefer a rule only if it improves net Sharpe or drawdown across more than one subperiod and does not rely on one extreme currency.",
    "- A positive G10 ATM interaction is useful only if the conditioned strategy improves the baseline after costs; the regression coefficient alone is not sufficient.",
    "- ATM volatility and butterfly should be judged primarily by risk reduction, drawdown, and expected shortfall, not only mean return.",
    "- Treat the expanding M5 ridge score as a diagnostic benchmark. A complex score that does not beat the simple hybrid after costs should not be promoted.",
    "- Parameter sensitivity should show a broad stable region. A single isolated best parameter is evidence of possible overfitting.",
])

display(Markdown("\n".join(interpretation)))

## Scope

This notebook performs transparent strategy construction and walk-forward diagnostics. It does not claim that the best full-sample strategy is production ready.

A production decision would additionally require:

- independently sourced transaction-cost estimates;
- forward execution timing and holiday alignment;
- liquidity and position limits;
- collateral and funding treatment;
- a truly untouched holdout period;
- live or paper-trading validation.